# ABAC Fundamentals

Attribute-Based Access Control (ABAC) primer on Databricks Unity Catalog —
row filters, column masks, and policy tags driven by user/group attributes.

## 1. RBAC, briefly

Role-Based Access Control is the model most enterprises still run on. It has three moving parts:

- **Users** — the humans (and service principals) who do work
- **Roles** — named bundles of permissions (`pharmacy_pricing_reader`, `store_manager_bc`, `payroll_admin`)
- **Permissions** — the actual grants on objects (SELECT on `sales.transactions`, MODIFY on `hr.payroll_runs`)

The contract is simple: a user is **assigned to a role**, the role **holds permissions**, end of story.

<!-- SVG: triangle diagram — User → Role → Permission, with arrows -->

### Why RBAC has been the default

RBAC stuck around for good reasons:

- **Auditable** — "who can do what" is a finite, listable set
- **Predictable** — adding a person to a role has obvious blast radius
- **Org-shaped** — roles map roughly to titles, which map roughly to org charts

It works **when the organization is small enough that the role list stays human-readable.** That assumption is where it falls apart.

## 2. The pain — role explosion

RBAC has a combinatorial problem. Every new dimension of access multiplies the role count.

At ARG, access decisions depend on at least:

| Dimension | Examples | Cardinality |
|---|---|---|
| Banner | FreshMart, Metro Kitchen, ValueMax, Harvest Market, Budget Basket | ~5–10 |
| Region | West Coast, Mountain, Central, Southeast, Northeast | ~6 |
| Store | Individual locations | ~150+ |
| Function | Pricing, payroll, inventory, loss prevention, pharmacy | ~10 |
| Sensitivity tier | Public, internal, confidential, regulated | 4 |
| Action | Read, write, approve, export | 4 |

A pure-RBAC encoding needs roles for every meaningful combination. Even a conservative cross-product is **tens of thousands of roles** — most of which exist for one person, get rubber-stamped at access reviews, and never get cleaned up.

In [0]:
displayHTML("""
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 980 450" font-family="Helvetica, Arial, sans-serif">
  <style>
    .title { font-size: 22px; font-weight: 700; fill: #1B3139; }
    .bar-rbac { fill: #1B3139; }
    .bar-rbac-top { fill: #FF3621; }
    .bar-abac { fill: #2EB67D; }
    .bar-number { font-size: 14px; font-weight: 700; fill: #1B3139; }
    .bar-number-top { font-size: 16px; font-weight: 700; fill: #FF3621; }
    .bar-number-abac { font-size: 14px; font-weight: 700; fill: #2EB67D; }
    .bar-label { font-size: 12px; fill: #1B3139; font-weight: 600; }
    .bar-sub { font-size: 11px; fill: #425563; font-style: italic; }
    .axis { stroke: #1B3139; stroke-width: 1.5; }
    .col-title { font-size: 14px; font-weight: 700; letter-spacing: 1.5px; fill: #1B3139; }
    .col-title-abac { font-size: 14px; font-weight: 700; letter-spacing: 1.5px; fill: #2EB67D; }
    .divider { stroke: #425563; stroke-width: 1; stroke-dasharray: 4 4; }
  </style>

  <text x="490" y="32" class="title" text-anchor="middle">Role explosion — each new dimension multiplies</text>

  <line class="axis" x1="40" y1="385" x2="900" y2="385"/>

  <text x="305" y="62" class="col-title" text-anchor="middle">PURE RBAC</text>
  <text x="820" y="62" class="col-title-abac" text-anchor="middle">ABAC</text>

  <rect class="bar-rbac" x="55" y="355" width="80" height="30"/>
  <text x="95" y="348" class="bar-number" text-anchor="middle">5</text>
  <text x="95" y="407" class="bar-label" text-anchor="middle">5 banners</text>

  <rect class="bar-rbac" x="160" y="305" width="80" height="80"/>
  <text x="200" y="298" class="bar-number" text-anchor="middle">30</text>
  <text x="200" y="407" class="bar-label" text-anchor="middle">× 6 regions</text>

  <rect class="bar-rbac" x="265" y="225" width="80" height="160"/>
  <text x="305" y="218" class="bar-number" text-anchor="middle">4,500</text>
  <text x="305" y="407" class="bar-label" text-anchor="middle">× 150 stores</text>

  <rect class="bar-rbac" x="370" y="165" width="80" height="220"/>
  <text x="410" y="158" class="bar-number" text-anchor="middle">45,000</text>
  <text x="410" y="407" class="bar-label" text-anchor="middle">× 10 functions</text>

  <rect class="bar-rbac-top" x="475" y="95" width="80" height="290"/>
  <text x="515" y="88" class="bar-number-top" text-anchor="middle">180,000</text>
  <text x="515" y="407" class="bar-label" text-anchor="middle">× 4 sensitivities</text>
  <text x="515" y="423" class="bar-sub" text-anchor="middle">(conservative cross-product)</text>

  <line class="divider" x1="635" y1="80" x2="635" y2="395"/>

  <rect class="bar-abac" x="780" y="365" width="80" height="20"/>
  <text x="820" y="358" class="bar-number-abac" text-anchor="middle">~5</text>
  <text x="820" y="407" class="bar-label" text-anchor="middle">policies, total</text>
  <text x="820" y="423" class="bar-sub" text-anchor="middle">dimensions live in attributes</text>
</svg>
""")

### The pain — joiners, movers, leavers

Every role is a row in a table someone has to maintain. The cost shows up in three lifecycle events:

- **Joiner** — new hire. Someone has to know which 8 roles the bundle actually needs. They usually copy from another employee, including the stale roles.
- **Mover** — promotion or transfer from FreshMart Riverside to Metro Kitchen downtown. Old roles stick around because **nobody removes; everyone adds**. This is how a department head ends up with read access to three banners they no longer work at.
- **Leaver** — termination. The roles deprovision, but if the leaver was the **only** person in a custom role, the role becomes orphaned junk.

The result is **permission creep**: actual access drifts further from intended access every quarter. Access reviews catch a fraction; the rest is invisible.

In [0]:
displayHTML("""
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 980 340" font-family="Helvetica, Arial, sans-serif">
  <style>
    .title { font-size: 22px; font-weight: 700; fill: #1B3139; }
    .axis { stroke: #1B3139; stroke-width: 1.5; }
    .staircase { fill: #1B3139; opacity: 0.88; }
    .cliff { stroke: #FF3621; stroke-width: 2; fill: none; stroke-dasharray: 4 3; }
    .event-line { stroke: #425563; stroke-width: 0.75; stroke-dasharray: 2 3; }
    .event-label { font-size: 13px; font-weight: 700; fill: #1B3139; }
    .event-delta { font-size: 11px; font-style: italic; fill: #425563; }
    .count-label { font-size: 13px; font-weight: 700; fill: #FFFFFF; }
    .orphan-label { font-size: 11px; font-weight: 700; fill: #FFFFFF; }
    .caption { font-size: 13px; fill: #425563; font-style: italic; }
  </style>

  <text x="490" y="32" class="title" text-anchor="middle">Joiner / Mover / Leaver — roles only ever accumulate</text>

  <line class="axis" x1="50" y1="290" x2="900" y2="290"/>

  <path class="staircase" d="M 80 290 L 80 270 L 280 270 L 280 240 L 440 240 L 440 200 L 600 200 L 600 170 L 760 170 L 760 290 Z"/>

  <line class="cliff" x1="760" y1="170" x2="760" y2="295"/>

  <line class="event-line" x1="80" y1="290" x2="80" y2="105"/>
  <line class="event-line" x1="280" y1="290" x2="280" y2="105"/>
  <line class="event-line" x1="440" y1="290" x2="440" y2="105"/>
  <line class="event-line" x1="600" y1="290" x2="600" y2="105"/>
  <line class="event-line" x1="760" y1="290" x2="760" y2="105"/>

  <text x="80" y="82" class="event-label" text-anchor="middle">Hire</text>
  <text x="80" y="98" class="event-delta" text-anchor="middle">+3 roles</text>

  <text x="280" y="82" class="event-label" text-anchor="middle">Promotion</text>
  <text x="280" y="98" class="event-delta" text-anchor="middle">+3 roles</text>

  <text x="440" y="82" class="event-label" text-anchor="middle">Banner Transfer</text>
  <text x="440" y="98" class="event-delta" text-anchor="middle">+4 roles · old stays</text>

  <text x="600" y="82" class="event-label" text-anchor="middle">Promotion</text>
  <text x="600" y="98" class="event-delta" text-anchor="middle">+3 roles</text>

  <text x="760" y="82" class="event-label" text-anchor="middle">Leaver</text>
  <text x="760" y="98" class="event-delta" text-anchor="middle">deprovisioned</text>

  <text x="180" y="286" class="count-label" text-anchor="middle">3</text>
  <text x="360" y="262" class="count-label" text-anchor="middle">6</text>
  <text x="520" y="225" class="count-label" text-anchor="middle">10</text>
  <text x="680" y="190" class="count-label" text-anchor="middle">13</text>

  <line stroke="#FF3621" stroke-width="1.5" stroke-dasharray="3 2" x1="780" y1="175" x2="770" y2="180"/>
  <rect x="780" y="155" width="160" height="40" rx="4" fill="#FF3621"/>
  <text x="860" y="172" class="orphan-label" text-anchor="middle">orphan role</text>
  <text x="860" y="187" class="orphan-label" text-anchor="middle" opacity="0.92">if leaver was sole owner</text>

  <text x="490" y="325" class="caption" text-anchor="middle">Roles accumulate across every life event. Access reviews catch a fraction. The rest is invisible.</text>
</svg>
""")

### The pain — granularity ceiling

RBAC grants are **set-shaped**, not **predicate-shaped**. You can grant SELECT on a table; you can’t natively grant “SELECT on rows where `store_id` matches my home store.”

The usual workarounds make things worse:

- **One view per role** — `sales_riverside_v`, `sales_oakfield_v`, `sales_greendale_v`. Now you have RBAC pain on roles **and** on views.
- **One schema per region** — duplicates pipelines and breaks cross-region analytics
- **Application-layer filtering** — moves the security perimeter into the BI tool, where auditors can’t see it

None of these scale. They just relocate the explosion.

## 3. The ABAC mental shift

ABAC stops asking *“who is this user?”* and starts asking **“what’s true about this access attempt right now?”**

Four things describe any access attempt:

| Term | What it is | Example |
|---|---|---|
| **Subject attributes** | Facts about the user | `department=Pharmacy`, `home_banner=FreshMart`, `clearance=Confidential` |
| **Resource attributes** | Facts about the object | `table.classification=PII`, `row.banner=Metro Kitchen`, `column.tag=SSN` |
| **Action attributes** | What’s being attempted | `SELECT`, `MODIFY`, `EXPORT` |
| **Environment attributes** | Context of the attempt | `time_of_day`, `network=corp_vpn`, `cluster.access_mode=shared` |

A policy is a sentence that combines these. The result is **decided at query time** — no role to mint, nothing to deprovision, nothing to drift.

In [0]:
displayHTML("""
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 900 420" font-family="Helvetica, Arial, sans-serif">
  <style>
    .title { font-size: 22px; font-weight: 700; fill: #1B3139; }
    .label { font-size: 13px; fill: #1B3139; }
    .label-white { font-size: 13px; fill: #FFFFFF; font-weight: 600; }
    .caption { font-size: 12px; fill: #425563; font-style: italic; }
    .shift-lbl { font-size: 14px; font-weight: 700; fill: #FF3621; letter-spacing: 1.5px; }
    .box { fill: #F9F7F4; stroke: #1B3139; stroke-width: 1.5; }
    .box-dark { fill: #1B3139; stroke: #1B3139; stroke-width: 1.5; }
    .box-brick { fill: #FF3621; stroke: #FF3621; stroke-width: 1.5; }
    .arrow { stroke: #425563; stroke-width: 1.5; fill: none; }
    .arrow-big { stroke: #FF3621; stroke-width: 3; fill: none; }
  </style>
  <defs>
    <marker id="ah" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto">
      <polygon points="0 0, 8 4, 0 8" fill="#425563"/>
    </marker>
    <marker id="ah-big" markerWidth="10" markerHeight="10" refX="9" refY="5" orient="auto">
      <polygon points="0 0, 10 5, 0 10" fill="#FF3621"/>
    </marker>
  </defs>

  <text x="150" y="40" class="title" text-anchor="middle">RBAC</text>

  <rect class="box" x="90" y="65" width="120" height="40" rx="6"/>
  <text x="150" y="90" class="label" text-anchor="middle">User</text>
  <line class="arrow" x1="150" y1="105" x2="150" y2="135" marker-end="url(#ah)"/>

  <rect class="box-dark" x="90" y="140" width="120" height="40" rx="6"/>
  <text x="150" y="165" class="label-white" text-anchor="middle">Role</text>
  <line class="arrow" x1="150" y1="180" x2="150" y2="210" marker-end="url(#ah)"/>

  <rect class="box" x="90" y="215" width="120" height="40" rx="6"/>
  <text x="150" y="240" class="label" text-anchor="middle">Permission</text>
  <line class="arrow" x1="150" y1="255" x2="150" y2="285" marker-end="url(#ah)"/>

  <rect class="box" x="90" y="290" width="120" height="40" rx="6"/>
  <text x="150" y="315" class="label" text-anchor="middle">Resource</text>

  <text x="150" y="365" class="caption" text-anchor="middle">Static — chain fixed</text>
  <text x="150" y="382" class="caption" text-anchor="middle">at assignment time.</text>

  <line class="arrow-big" x1="240" y1="200" x2="430" y2="200" marker-end="url(#ah-big)"/>
  <text x="335" y="190" class="shift-lbl" text-anchor="middle">SHIFT</text>

  <text x="665" y="40" class="title" text-anchor="middle">ABAC</text>

  <rect class="box" x="470" y="72" width="130" height="36" rx="6"/>
  <text x="535" y="95" class="label" text-anchor="middle">Subject attrs</text>

  <rect class="box" x="470" y="122" width="130" height="36" rx="6"/>
  <text x="535" y="145" class="label" text-anchor="middle">Resource attrs</text>

  <rect class="box" x="470" y="172" width="130" height="36" rx="6"/>
  <text x="535" y="195" class="label" text-anchor="middle">Action</text>

  <rect class="box" x="470" y="222" width="130" height="36" rx="6"/>
  <text x="535" y="245" class="label" text-anchor="middle">Environment</text>

  <path class="arrow" d="M 600 90 C 625 90, 630 175, 645 178" marker-end="url(#ah)"/>
  <path class="arrow" d="M 600 140 C 615 140, 630 175, 645 178" marker-end="url(#ah)"/>
  <path class="arrow" d="M 600 190 C 615 190, 630 182, 645 182" marker-end="url(#ah)"/>
  <path class="arrow" d="M 600 240 C 625 240, 630 185, 645 182" marker-end="url(#ah)"/>

  <rect class="box-brick" x="650" y="145" width="120" height="70" rx="6"/>
  <text x="710" y="175" class="label-white" text-anchor="middle">Policy</text>
  <text x="710" y="195" class="label-white" text-anchor="middle">Engine</text>

  <line class="arrow" x1="770" y1="180" x2="810" y2="180" marker-end="url(#ah)"/>

  <rect class="box-dark" x="815" y="155" width="70" height="50" rx="6"/>
  <text x="850" y="175" class="label-white" text-anchor="middle">Allow</text>
  <text x="850" y="195" class="label-white" text-anchor="middle">/ Mask</text>

  <text x="665" y="365" class="caption" text-anchor="middle">Dynamic — decision</text>
  <text x="665" y="382" class="caption" text-anchor="middle">computed at query time.</text>
</svg>
""")

## 4. Anatomy of an ABAC policy

A policy is shaped like:

> **Allow** *action* on *resource* **when** *condition over attributes*

Examples in plain English, expressed against our retail data:

- *Allow* SELECT on `sales.transactions` *when* `user.home_banner == row.banner`
- *Mask* column `customer.email` *when* `user.clearance < 'Confidential'`
- *Allow* MODIFY on `hr.payroll_runs` *when* `user.department == 'Payroll'` **and** `environment.network == 'corp_vpn'`

Notice what’s **not** in those policies: no role names, no store IDs hard-coded, no user emails. The policy is a **rule**, and the rule applies as facts change.

One policy replaces the cross-product of roles it would have taken to express the same intent.

In [0]:
displayHTML("""
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 900 290" font-family="Helvetica, Arial, sans-serif">
  <style>
    .title { font-size: 20px; font-weight: 700; fill: #1B3139; }
    .static-text { font-size: 17px; font-family: 'Courier New', Courier, monospace; fill: #1B3139; font-weight: 600; }
    .pill-text { font-size: 15px; font-family: 'Courier New', Courier, monospace; fill: #FFFFFF; font-weight: 700; }
    .legend { font-size: 12px; fill: #425563; font-style: italic; }
    .callout-line { stroke: #425563; stroke-width: 1; fill: none; stroke-dasharray: 2 2; }
    .callout-label { font-size: 13px; fill: #1B3139; font-weight: 700; }
    .callout-sub { font-size: 11px; fill: #425563; }
  </style>

  <text x="450" y="32" class="title" text-anchor="middle">Anatomy of an ABAC policy</text>

  <text x="40" y="98" class="legend">template</text>

  <rect x="120" y="80" width="80" height="30" rx="15" fill="#2EB67D"/>
  <text x="160" y="100" class="pill-text" text-anchor="middle">ALLOW</text>

  <rect x="210" y="80" width="100" height="30" rx="15" fill="#1B3139"/>
  <text x="260" y="100" class="pill-text" text-anchor="middle">[action]</text>

  <text x="320" y="100" class="static-text">on</text>

  <rect x="360" y="80" width="180" height="30" rx="15" fill="#FF3621"/>
  <text x="450" y="100" class="pill-text" text-anchor="middle">[resource]</text>

  <text x="550" y="100" class="static-text">when</text>

  <rect x="615" y="80" width="260" height="30" rx="15" fill="#00A1C9"/>
  <text x="745" y="100" class="pill-text" text-anchor="middle">[condition over attributes]</text>

  <text x="40" y="168" class="legend">example</text>

  <rect x="120" y="150" width="80" height="30" rx="15" fill="#2EB67D"/>
  <text x="160" y="170" class="pill-text" text-anchor="middle">ALLOW</text>

  <rect x="210" y="150" width="100" height="30" rx="15" fill="#1B3139"/>
  <text x="260" y="170" class="pill-text" text-anchor="middle">SELECT</text>

  <text x="320" y="170" class="static-text">on</text>

  <rect x="360" y="150" width="180" height="30" rx="15" fill="#FF3621"/>
  <text x="450" y="170" class="pill-text" text-anchor="middle">sales.transactions</text>

  <text x="550" y="170" class="static-text">when</text>

  <rect x="615" y="150" width="260" height="30" rx="15" fill="#00A1C9"/>
  <text x="745" y="170" class="pill-text" text-anchor="middle">user.banner = row.banner</text>

  <line class="callout-line" x1="160" y1="185" x2="160" y2="210"/>
  <text x="160" y="226" class="callout-label" text-anchor="middle">Effect</text>
  <text x="160" y="244" class="callout-sub" text-anchor="middle">Allow / Deny</text>
  <text x="160" y="258" class="callout-sub" text-anchor="middle">/ Mask</text>

  <line class="callout-line" x1="260" y1="185" x2="260" y2="210"/>
  <text x="260" y="226" class="callout-label" text-anchor="middle">Action</text>
  <text x="260" y="244" class="callout-sub" text-anchor="middle">What's being</text>
  <text x="260" y="258" class="callout-sub" text-anchor="middle">attempted</text>

  <line class="callout-line" x1="450" y1="185" x2="450" y2="210"/>
  <text x="450" y="226" class="callout-label" text-anchor="middle">Resource</text>
  <text x="450" y="244" class="callout-sub" text-anchor="middle">Object identity</text>
  <text x="450" y="258" class="callout-sub" text-anchor="middle">or tag match</text>

  <line class="callout-line" x1="745" y1="185" x2="745" y2="210"/>
  <text x="745" y="226" class="callout-label" text-anchor="middle">Condition</text>
  <text x="745" y="244" class="callout-sub" text-anchor="middle">Predicate over subject,</text>
  <text x="745" y="258" class="callout-sub" text-anchor="middle">resource, env attributes</text>
</svg>
""")

## 5. RBAC vs ABAC at a glance

| | RBAC | ABAC |
|---|---|---|
| **Unit of policy** | Role | Predicate over attributes |
| **Granted to** | Users (via role assignment) | Nobody — evaluated per request |
| **Adapts when facts change** | No — admin must re-grant | Yes — attribute change propagates |
| **Row-level filtering** | Workaround (views, schemas) | First-class |
| **Column masking** | Workaround | First-class |
| **Audit story** | "Who is in role X?" | "Why was this row returned?" |
| **Failure mode** | Permission creep | Bad attribute hygiene |

ABAC doesn't eliminate operational burden — it **moves** it. The new burden is keeping **attributes** accurate (HR feed, group membership, table tags). That's an easier problem because attributes already have owners; roles often don't.

<!-- SVG: side-by-side comparison panels — RBAC structure on the left, ABAC structure on the right -->

## 6. Where ABAC lives in Databricks Unity Catalog

Unity Catalog gives you the primitives to do ABAC without building it yourself:

- **Subject attributes** → user identity + **group membership** (synced from IdP) + custom group properties
- **Resource attributes** → **tags** on catalogs, schemas, tables, and columns
- **Action attributes** → built into the grant model (SELECT, MODIFY, etc.)
- **Environment attributes** → cluster access mode, workspace, network

The two enforcement points we'll spend the rest of the notebook on:

- **Row filters** — SQL UDF that returns a boolean; runs per row at query time
- **Column masks** — SQL UDF that transforms a column value based on caller attributes

Both are written **once per table**, evaluated **per query**, and **invisible to the user** writing the SQL. That's the ABAC payoff: the model centralizes the policy and decentralizes the decision.

In [0]:
displayHTML("""
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 980 500" font-family="Helvetica, Arial, sans-serif">
  <style>
    .title { font-size: 24px; font-weight: 700; fill: #1B3139; }
    .attr-title-green { font-size: 16px; font-weight: 700; fill: #2EB67D; letter-spacing: 1.2px; }
    .attr-title-brick { font-size: 16px; font-weight: 700; fill: #FF3621; letter-spacing: 1.2px; }
    .attr-title-navy  { font-size: 16px; font-weight: 700; fill: #1B3139; letter-spacing: 1.2px; }
    .attr-title-cyan  { font-size: 16px; font-weight: 700; fill: #00A1C9; letter-spacing: 1.2px; }
    .attr-sub { font-size: 12px; fill: #425563; }
    .pipe-label { font-size: 16px; fill: #1B3139; font-weight: 600; }
    .pipe-label-white { font-size: 16px; fill: #FFFFFF; font-weight: 700; }
    .pipe-sub-white { font-size: 12px; fill: #FFFFFF; opacity: 0.85; }
    .pipe-caption { font-size: 11px; fill: #425563; font-style: italic; letter-spacing: 1.5px; }
    .box-soft { fill: #F9F7F4; stroke: #1B3139; stroke-width: 1.5; }
    .box-dark { fill: #1B3139; stroke: #1B3139; stroke-width: 1.5; }
    .arrow { stroke: #425563; stroke-width: 1.5; fill: none; }
    .arrow-thin { stroke: #425563; stroke-width: 1; fill: none; stroke-dasharray: 4 3; }
  </style>
  <defs>
    <marker id="ah" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto">
      <polygon points="0 0, 8 4, 0 8" fill="#425563"/>
    </marker>
  </defs>

  <text x="490" y="38" class="title" text-anchor="middle">Where ABAC lives in Databricks Unity Catalog</text>

  <rect class="box-soft" x="35" y="80" width="270" height="80" rx="6"/>
  <text x="170" y="112" class="attr-title-green" text-anchor="middle">SUBJECT</text>
  <text x="170" y="132" class="attr-sub" text-anchor="middle">User identity + group memberships</text>
  <text x="170" y="148" class="attr-sub" text-anchor="middle">(synced from IdP)</text>

  <rect class="box-soft" x="320" y="80" width="640" height="80" rx="6"/>
  <text x="640" y="112" class="attr-title-brick" text-anchor="middle">RESOURCE</text>
  <text x="640" y="138" class="attr-sub" text-anchor="middle">Tags on catalog / schema / table / column</text>

  <text x="490" y="217" class="pipe-caption" text-anchor="middle">POLICY EVALUATION — PER QUERY</text>

  <rect class="box-soft" x="50" y="235" width="240" height="80" rx="6"/>
  <text x="170" y="280" class="pipe-label" text-anchor="middle">UC Grants Check</text>

  <line class="arrow" x1="290" y1="275" x2="318" y2="275" marker-end="url(#ah)"/>

  <rect class="box-dark" x="320" y="235" width="280" height="80" rx="6"/>
  <text x="460" y="270" class="pipe-label-white" text-anchor="middle">Row Filter</text>
  <text x="460" y="294" class="pipe-sub-white" text-anchor="middle">SQL UDF</text>

  <line class="arrow" x1="600" y1="275" x2="628" y2="275" marker-end="url(#ah)"/>

  <rect class="box-dark" x="630" y="235" width="280" height="80" rx="6"/>
  <text x="770" y="270" class="pipe-label-white" text-anchor="middle">Column Mask</text>
  <text x="770" y="294" class="pipe-sub-white" text-anchor="middle">SQL UDF</text>

  <rect class="box-soft" x="35" y="395" width="270" height="80" rx="6"/>
  <text x="170" y="424" class="attr-title-navy" text-anchor="middle">ACTION</text>
  <text x="170" y="446" class="attr-sub" text-anchor="middle">SELECT, MODIFY, EXPORT</text>
  <text x="170" y="462" class="attr-sub" text-anchor="middle">(built into the grant model)</text>

  <rect class="box-soft" x="320" y="395" width="640" height="80" rx="6"/>
  <text x="640" y="424" class="attr-title-cyan" text-anchor="middle">ENVIRONMENT</text>
  <text x="640" y="450" class="attr-sub" text-anchor="middle">Cluster access mode, workspace, network</text>

  <line class="arrow-thin" x1="170" y1="161" x2="170" y2="233" marker-end="url(#ah)"/>
  <line class="arrow-thin" x1="170" y1="394" x2="170" y2="317" marker-end="url(#ah)"/>

  <line class="arrow-thin" x1="460" y1="161" x2="460" y2="233" marker-end="url(#ah)"/>
  <line class="arrow-thin" x1="770" y1="161" x2="770" y2="233" marker-end="url(#ah)"/>

  <line class="arrow-thin" x1="460" y1="394" x2="460" y2="317" marker-end="url(#ah)"/>
  <line class="arrow-thin" x1="770" y1="394" x2="770" y2="317" marker-end="url(#ah)"/>
</svg>
""")

# Section 2 — Auto-Tagging Pipeline

The ABAC theory from Section 1 only works if your **resource attributes** are populated. Tagging hygiene is what makes that real. Databricks gives you two tagging streams:

1. **Automatic** — an AI agent classifies columns for sensitive data and applies governed system tags
2. **Manual** — domain teams apply custom governed tags (banner, data owner, sensitivity tier)

We’ll cover both, then run a live demo against sample retail data.

## What Databricks Data Classification does

- **AI-driven column scanner.** An agent samples each column and detects PII patterns — SSN, email, phone, street address, credit card, government ID, more.
- **Incremental, no manual config.** New tables in a classification-enabled catalog get scanned within 24 hours. Existing tables get re-scanned on schedule.
- **Applies governed tags.** Results show up as tag key-value pairs like `class_pii: true` and `class_pii_type: email` directly on the columns it identified.
- **No data movement.** Sampling and inference happen inside your workspace — sensitive values don't leave the UC perimeter.

The output is the raw material for **resource attributes** in your ABAC policies. Row filters and column masks read these tags to decide what to filter or mask.

## System tags + custom governed tags

Two flavors of tags coexist on the same UC objects:

| | Applied by | Examples | Use case |
|---|---|---|---|
| **System tags** | Data Classification agent | `class_pii: true`, `class_pii_type: ssn` | Auto-discovered sensitivity |
| **Custom governed tags** | Domain teams (manual) | `banner: freshmart`, `data_owner: supply_chain`, `sensitivity: internal` | Business context UC can’t infer |

Both are first-class. Both can be read by row filters and column masks. The key is **discipline** — once tags become policy inputs, drift in tagging becomes drift in access.

In [0]:
displayHTML("""
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 980 460" font-family="Helvetica, Arial, sans-serif">
  <style>
    .title { font-size: 22px; font-weight: 700; fill: #1B3139; }
    .stream-title-auto { font-size: 16px; font-weight: 700; letter-spacing: 1px; fill: #00A1C9; }
    .stream-title-manual { font-size: 16px; font-weight: 700; letter-spacing: 1px; fill: #2EB67D; }
    .stream-sub { font-size: 12px; fill: #425563; font-style: italic; }
    .tag-mono { font-size: 11px; font-family: 'Courier New', Courier, monospace; fill: #1B3139; }
    .center-label-white { font-size: 16px; font-weight: 700; fill: #FFFFFF; }
    .center-sub-white { font-size: 12px; fill: #FFFFFF; opacity: 0.88; }
    .box-soft { fill: #F9F7F4; stroke: #1B3139; stroke-width: 1.5; }
    .box-brick { fill: #FF3621; stroke: #FF3621; stroke-width: 1.5; }
    .box-dark { fill: #1B3139; stroke: #1B3139; stroke-width: 1.5; }
    .arrow-auto { stroke: #00A1C9; stroke-width: 2; fill: none; }
    .arrow-manual { stroke: #2EB67D; stroke-width: 2; fill: none; }
    .arrow-gray { stroke: #425563; stroke-width: 2; fill: none; }
  </style>
  <defs>
    <marker id="ah-auto" markerWidth="10" markerHeight="10" refX="9" refY="5" orient="auto">
      <polygon points="0 0, 10 5, 0 10" fill="#00A1C9"/>
    </marker>
    <marker id="ah-manual" markerWidth="10" markerHeight="10" refX="9" refY="5" orient="auto">
      <polygon points="0 0, 10 5, 0 10" fill="#2EB67D"/>
    </marker>
    <marker id="ah-gray2" markerWidth="10" markerHeight="10" refX="9" refY="5" orient="auto">
      <polygon points="0 0, 10 5, 0 10" fill="#425563"/>
    </marker>
  </defs>

  <text x="490" y="32" class="title" text-anchor="middle">Two tagging streams, one tagged-object surface</text>

  <rect class="box-soft" x="70" y="65" width="260" height="100" rx="6"/>
  <text x="200" y="90" class="stream-title-auto" text-anchor="middle">AUTOMATIC</text>
  <text x="200" y="108" class="stream-sub" text-anchor="middle">Data Classification agent</text>
  <text x="200" y="135" class="tag-mono" text-anchor="middle">class_pii: true</text>
  <text x="200" y="151" class="tag-mono" text-anchor="middle">class_pii_type: email|phone|ssn…</text>

  <rect class="box-soft" x="650" y="65" width="260" height="100" rx="6"/>
  <text x="780" y="90" class="stream-title-manual" text-anchor="middle">MANUAL</text>
  <text x="780" y="108" class="stream-sub" text-anchor="middle">Domain teams (CoE)</text>
  <text x="780" y="135" class="tag-mono" text-anchor="middle">banner, data_owner</text>
  <text x="780" y="151" class="tag-mono" text-anchor="middle">sensitivity, business_unit</text>

  <path class="arrow-auto" d="M 220 168 C 280 200, 380 220, 410 240" marker-end="url(#ah-auto)"/>
  <path class="arrow-manual" d="M 760 168 C 700 200, 600 220, 570 240" marker-end="url(#ah-manual)"/>

  <rect class="box-brick" x="290" y="245" width="400" height="80" rx="6"/>
  <text x="490" y="275" class="center-label-white" text-anchor="middle">Unity Catalog objects</text>
  <text x="490" y="298" class="center-sub-white" text-anchor="middle">catalog · schema · table · column</text>
  <text x="490" y="316" class="center-sub-white" text-anchor="middle">carry both system + custom tags</text>

  <line class="arrow-gray" x1="490" y1="325" x2="490" y2="365" marker-end="url(#ah-gray2)"/>

  <rect class="box-dark" x="290" y="370" width="400" height="60" rx="6"/>
  <text x="490" y="395" class="center-label-white" text-anchor="middle">Row filters + Column masks</text>
  <text x="490" y="416" class="center-sub-white" text-anchor="middle">read these tags at query time</text>
</svg>
""")

## What this looks like for a multi-banner retailer

As your legacy ERP wage data, POS transactions, and customer loyalty data land in Unity Catalog, the classification agent catches the sensitive columns automatically — **no manual tagging effort**

Day-zero: emails, phone numbers, payroll amounts, loyalty card IDs get tagged as PII without anyone filing a ticket.

Day-one onward: The domain owners layer on the *domain* tags — which banner, which data owner, which sensitivity tier — that govern who can see what. Those are the tags the row filters and column masks we’ll write in the next section will read.

## Live demo — tagging a retail transactions table

In this demo we will:

1. Spin up a sample transactions table in a sandbox schema
2. Show what auto-classification **would** apply (simulating it inline, since the agent runs on a 24h cadence)
3. Layer on the custom governed tags
4. Query `information_schema` to see the full tagging state

In [0]:
dbutils.widgets.text("catalog", "main", "Catalog")
dbutils.widgets.text("schema",  "abac_demo", "Schema")

In [0]:
%sql
-- Create the sandbox catalog (idempotent)
-- CREATE CATALOG IF NOT EXISTS IDENTIFIER(:catalog);

In [0]:
%sql
USE CATALOG IDENTIFIER(:catalog);

In [0]:
%sql
-- Create the demo schema inside the current catalog
-- CREATE SCHEMA IF NOT EXISTS IDENTIFIER(:schema);

In [0]:
%sql
USE SCHEMA IDENTIFIER(:schema);

In [0]:
%sql
-- Sample retail transactions table.  Notice the mix:
--   * Banner / store / region   → row filter inputs
--   * email / phone / loyalty   → column mask candidates (PII)
--   * employee id               → PII for staff
CREATE OR REPLACE TABLE transactions (
  transaction_id    BIGINT,
  banner            STRING        COMMENT 'Retail banner name',
  store_id          STRING        COMMENT 'Store identifier',
  store_region      STRING        COMMENT 'State / region',
  customer_email    STRING        COMMENT 'Customer contact email',
  customer_phone    STRING        COMMENT 'Customer contact phone',
  loyalty_card_id   STRING        COMMENT 'Rewards+ loyalty card identifier',
  cashier_emp_id    STRING        COMMENT 'Employee ID of cashier',
  payment_method    STRING,
  total_amount      DECIMAL(10,2),
  transaction_ts    TIMESTAMP
);

In [0]:
%sql
INSERT INTO transactions VALUES
  (1001, 'FreshMart', 'CA-RIVERSIDE-014',    'CA', 'jane.kim@example.com',    '626-555-0142', 'FM-9931-2204', 'EMP-08812', 'debit',     87.42,  TIMESTAMP '2026-05-10 17:22:11'),
  (1002, 'FreshMart', 'CA-LAKEWOOD-001', 'CA', 'mark.wei@example.com',    '626-555-0188', 'FM-1102-8856', 'EMP-04217', 'visa',      142.15, TIMESTAMP '2026-05-10 18:01:44'),
  (1003, 'Metro Kitchen',    'CA-MIDTOWN-003',  'CA', 'priya.menon@example.com', '626-555-0103', 'MK-7700-5512',  'EMP-01199', 'amex',      54.20,  TIMESTAMP '2026-05-10 11:14:02'),
  (1004, 'ValueMax',    'CA-GREENDALE-007',   'CA', 'cathy.singh@example.com', '626-555-0177', 'VM-2233-0188',  'EMP-09012', 'cash',      211.95, TIMESTAMP '2026-05-10 15:48:31'),
  (1005, 'Harvest Market', 'WA-PINECREST-002',     'WA', 'tom.barber@example.com',  '425-555-0144', 'HM-4471-3309',  'EMP-03114', 'debit',     76.10,  TIMESTAMP '2026-05-10 13:55:09'),
  (1006, 'FreshMart', 'TX-AUSTIN-022',   'TX', 'linda.zhao@example.com',  '512-555-0156', 'FM-5511-8821', 'EMP-07731', 'visa',      38.75,  TIMESTAMP '2026-05-10 09:32:17'),
  (1007, 'Metro Kitchen',    'CA-DOWNTOWN-001',  'CA', 'derek.olu@example.com',   '626-555-0199', 'MK-3308-9941',  'EMP-02201', 'apple_pay', 92.30,  TIMESTAMP '2026-05-10 19:11:50'),
  (1008, 'Budget Basket', 'WA-WESTFIELD-005',  'WA', 'eve.ng@example.com',      '425-555-0125', 'BB-1188-6677',  'EMP-05509', 'debit',     168.40, TIMESTAMP '2026-05-10 14:27:33');

In [0]:
%sql
SELECT * FROM transactions LIMIT 5;

### Step 1 — What auto-classification would apply

In a real classification-enabled catalog, the agent would tag the sensitive columns within 24h. To keep the demo flowing, we'll apply the same tags the agent would set — `class.pii_type` on each PII column.

In [0]:
%sql
-- Simulating what Data Classification would auto-apply.
-- Note: UC reserves `.`, `=`, `/` in tag keys, so we use underscored keys.
ALTER TABLE transactions ALTER COLUMN customer_email  SET TAGS ('class_pii' = 'true', 'class_pii_type' = 'email');
ALTER TABLE transactions ALTER COLUMN customer_phone  SET TAGS ('class_pii' = 'true', 'class_pii_type' = 'phone');
ALTER TABLE transactions ALTER COLUMN loyalty_card_id SET TAGS ('class_pii' = 'true', 'class_pii_type' = 'loyalty_id');
ALTER TABLE transactions ALTER COLUMN cashier_emp_id  SET TAGS ('class_pii' = 'true', 'class_pii_type' = 'employee_id');

In [0]:
%sql
-- The agent's output, surfaced via information_schema
SELECT column_name, tag_name, tag_value
FROM   system.information_schema.column_tags
WHERE  catalog_name = :catalog
  AND  schema_name  = :schema
  AND  table_name   = 'transactions'
ORDER BY column_name, tag_name;

### Step 2 — Layer on the custom governed tags

Auto-classification handles *what kind of data this is*. The domain team layers on *who owns it* and *which slice of the business it belongs to*. These tags are what the row filter will read.

In [0]:
%sql
-- Note: this workspace has a tag policy on `sensitivity` enforcing the
-- vocabulary [pii, internal, public] — exactly the kind of guardrail
-- you want once tags drive access policy.
ALTER TABLE transactions SET TAGS (
  'data_owner'    = 'supply_chain',
  'sensitivity'   = 'internal',
  'business_unit' = 'Operations'
);

-- Mark `banner` as the row-filter key so the policy author knows where to hook
ALTER TABLE transactions ALTER COLUMN banner SET TAGS ('row_filter_key' = 'true');

In [0]:
%sql
-- Combined view: every tag on the table, system + custom, table-level + column-level
SELECT 'TABLE'                       AS scope, tag_name, tag_value
FROM   system.information_schema.table_tags
WHERE  catalog_name = :catalog AND schema_name = :schema AND table_name = 'transactions'

UNION ALL

SELECT CONCAT('COL: ', column_name)  AS scope, tag_name, tag_value
FROM   system.information_schema.column_tags
WHERE  catalog_name = :catalog AND schema_name = :schema AND table_name = 'transactions'

ORDER BY scope, tag_name;

## Recap

We just produced the **resource attributes** the ABAC engine needs:

- **PII columns** are tagged automatically — `class_pii_type: email/phone/loyalty_id/employee_id`
- **Domain context** is tagged manually — `banner`, `data_owner`, `sensitivity`, `business_unit`
- **Policy hooks** are marked explicitly — `row_filter_key` on the column the next-section filter will read

Next section we'll write the **row filter** and **column mask** UDFs that consume these tags. The policy logic stays simple because the tags do the heavy lifting.

## Bonus — setting up your own tag policies

### What is a tag policy?

A tag policy is a metastore-level governance artifact that locks down which **values** are allowed for a given tag **key**. Without one, anyone with tag-write permission can apply any value. With one, UC rejects out-of-vocabulary values at the `ALTER TABLE` boundary — *before* the bad tag ever reaches a row filter.

### Why this matters for ABAC

Once a tag drives access (e.g. a row filter reads `sensitivity` to decide what to mask), **any drift in tag values is silent drift in access policy**. One analyst tagging `Confidential!` instead of `confidential` opens a gap nobody notices until the audit. Tag policies are the upstream guardrail that prevents that drift from ever landing.

### Where to manage them

As of current Databricks releases, tag policies are managed primarily through:

- **UI** — Catalog Explorer → *Governance* → *Tag Policies*. Easiest path; this is where the `sensitivity` and `business_unit` policies you hit live were created.
- **REST API** — `/api/2.1/unity-catalog/tag-policies` for programmatic setup and IaC (Terraform via the Databricks provider).

A stable SQL DDL (`CREATE TAG POLICY`) for this is on the roadmap but isn’t reliably available as of writing — check current docs before scripting. The practical path today is: pilot vocabulary in the UI, then once stable, lift the same definitions into Terraform.

### A pragmatic rollout

Start narrow. Three or four governed keys is enough to anchor the program — every key you add to a policy is a key your ABAC policies can trust.

| Tag key | Suggested vocabulary | Why this one first |
|---|---|---|
| `sensitivity` | `pii`, `internal`, `public` | Drives column masks and row filters directly |
| `data_owner` | curated list of team identifiers | Routes every table to an accountable owner |
| `business_unit` | `Finance`, `Operations`, `Pharmacy`, `Loyalty`, `Supply Chain` | Maps tables to their analytical home; aids cost/usage attribution |
| `banner` | `FreshMart`, `Metro Kitchen`, `ValueMax`, `Harvest Market`, `Budget Basket` | The dominant row-filter dimension for retail data |

Iterate from there. Roll out tags first (ungoverned), watch what values actually land in the wild, *then* lock the vocabulary based on real usage rather than top-down guesses.

### Who owns the policy?

Tag-policy creation requires metastore-admin privileges. In practice that’s a small group — typically the central data governance team. The *values* inside each policy should be agreed cross-functionally (a banner list is meaningless without retail input, an owner list is meaningless without finance/HR input). Treat the policy DDL like any other infra change: code-reviewed, version-controlled, applied via CI.

# Section 3 — Building & Applying Policies

Tags give us the **resource attributes**; the section above set those up. Now we'll write the actual ABAC enforcement. Two policy types do the work — both are SQL UDFs you attach to UC objects.

## Two policy types

| | What it does | Return type | Example |
|---|---|---|---|
| **Row Filter** | Decides whether a row is visible to the caller | `BOOLEAN` | "Return TRUE only when `banner = caller's home banner`" |
| **Column Mask** | Transforms a column's value before returning | Same type as the column | "Return last 4 of SSN unless caller is in `hr_admins`" |

Both can be attached at **table** level (specific column/table) *or* **catalog** level (any column matching a tag predicate). The catalog-level pattern is where ABAC pays off — tag a column once, the policy applies everywhere automatically.

## Policy anatomy

Every policy has four moving parts:

1. **Scope** — which UC object the policy attaches to (a single table, a catalog, or all columns with a given tag in a catalog)
2. **Match condition** — usually a tag predicate, e.g. *"this column has tag `pii = ssn`"*
3. **Caller condition** — who's exempt vs subject to the policy (group membership, attribute checks via `is_account_group_member()`, `current_user()`)
4. **Transformation** — the SQL UDF that returns the boolean (row filter) or transformed value (column mask)

The UDF itself is *just SQL* — `CASE WHEN ... THEN ... ELSE ... END`. If you can write a `CASE`, you can write an ABAC policy.

## Live demo — column mask via tag, attached at catalog scope

Following the official ABAC tutorial pattern:

1. Define a governed tag `pii` with allowed values `ssn`, `address`
2. Create a small HR-shaped `employees` table with sensitive columns
3. Tag those columns with `pii = ssn` / `pii = address`
4. Write mask UDFs that hide the value unless the caller is in `hr_admins`
5. Attach the masks at **catalog** scope — anything tagged `pii = ssn` in the catalog gets masked, table by table or column by column, automatically
6. Query and observe the mask firing

**The key moment** is step 5 → step 6: we'll create a *second* table in the same catalog, tag a column, and watch the mask apply with zero per-table wiring.

In [0]:
%sql
-- Who am I, and what groups am I in?
-- The mask UDFs we're about to write will use this to decide whether to redact.
SELECT
  current_user()                            AS me,
  is_account_group_member('hr_admins')      AS am_i_hr_admin;

### Step 1 — Govern the `pii` tag vocabulary

Same pattern we covered in the Bonus section of Section 2 — lock the allowed values before anyone applies the tag in anger.

**Note on tag-policy creation:**

In current Databricks releases, tag policies are managed via the **Catalog Explorer UI** (*Governance → Tag Policies*) or the **REST API** — there isn't yet a stable SQL DDL surface for `CREATE TAG POLICY`. We're skipping the create step here for that reason; the rest of the demo works regardless (an ungoverned tag key just accepts any value). We will be setting up a catalog-level policy later.

If you want to set up the `pii` vocabulary in this workspace before the audience arrives, do it once in the UI — same pattern you saw on the pre-existing `sensitivity` / `business_unit` policies in Section 2.

### Step 2 — A small HR-shaped table

In [0]:
%sql
DROP TABLE IF EXISTS employees;
CREATE OR REPLACE TABLE employees (
  employee_id   STRING,
  full_name     STRING,
  banner        STRING,
  store_id      STRING,
  ssn           STRING   COMMENT 'Social security number — sensitive',
  home_address  STRING   COMMENT 'Residential address — sensitive',
  hire_date     DATE
);

In [0]:
%sql
INSERT INTO employees VALUES
  ('EMP-08812', 'Jane Kim',     'FreshMart', 'CA-RIVERSIDE-014',    '111-22-3344', '1212 Maple St, Riverside CA',       DATE '2019-03-15'),
  ('EMP-04217', 'Mark Wei',     'FreshMart', 'CA-LAKEWOOD-001', '222-33-4455', '88 Oak Ave, Lakewood CA',       DATE '2020-07-22'),
  ('EMP-01199', 'Priya Menon',  'Metro Kitchen',    'CA-MIDTOWN-003',  '333-44-5566', '405 Elm St, Lakewood CA', DATE '2021-11-08'),
  ('EMP-09012', 'Cathy Singh',  'ValueMax',    'CA-GREENDALE-007',   '444-55-6677', '900 Main Rd, Greendale CA',      DATE '2018-05-30'),
  ('EMP-03114', 'Tom Barber',   'Harvest Market', 'WA-PINECREST-002',     '555-66-7788', '77 Pine Rd, Pinecrest WA',          DATE '2022-01-12');

In [0]:
%sql
-- Sanity check — raw data, no masks yet
SELECT * FROM employees;

### Step 3 — Tag the sensitive columns

In [0]:
%sql
ALTER TABLE employees ALTER COLUMN ssn          SET TAGS ('pii' = 'ssn');
ALTER TABLE employees ALTER COLUMN home_address SET TAGS ('pii' = 'address');

### Step 4 — Write the mask UDFs

Pure SQL. The CASE is doing all the work — `hr_admins` see the raw value, everyone else sees the redacted form.

In [0]:
%sql
-- SSN: hr_admins see raw; everyone else sees `XXX-XX-<last 4>`
CREATE OR REPLACE FUNCTION mask_ssn(val STRING)
RETURNS STRING
RETURN CASE
  WHEN is_account_group_member('hr_admins') THEN val
  WHEN val IS NULL                          THEN NULL
  ELSE CONCAT('XXX-XX-', RIGHT(val, 4))
END;

In [0]:
%sql
-- Address: hr_admins see raw; everyone else sees `[REDACTED — <province>]`
CREATE OR REPLACE FUNCTION mask_address(val STRING)
RETURNS STRING
RETURN CASE
  WHEN is_account_group_member('hr_admins') THEN val
  WHEN val IS NULL                          THEN NULL
  ELSE CONCAT('[REDACTED — ', RIGHT(val, 2), ']')
END;

### Step 5 — Attach the masks

Two ways to attach a column mask in Unity Catalog today:

| Scope | How it's attached | What it gives you |
|---|---|---|
| **Per table / column** (what we'll demo) | `ALTER TABLE ... ALTER COLUMN ... SET MASK function_name` — stable SQL DDL | The mask runs whenever this specific column is queried |
| **Catalog-scope, tag-driven** (the ABAC end state) | Currently **UI / REST API** — Catalog Explorer → *Governance* → *Policies* | Any column anywhere in the catalog that carries the matching tag picks up the mask automatically — no per-table wiring |

We're going to use the per-table form for the live demo because the SQL is rock solid. The audience can imagine the catalog-scope form trivially — *"same UDF, attached once, applies everywhere a column is tagged `pii = ssn`."*

In [0]:
displayHTML("""
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1000 480" font-family="Helvetica, Arial, sans-serif">
  <style>
    .title { font-size: 22px; font-weight: 700; fill: #1B3139; }
    .panel-title { font-size: 16px; font-weight: 700; fill: #1B3139; letter-spacing: 1.5px; }
    .panel-sub { font-size: 12px; fill: #425563; font-style: italic; }
    .box-label { font-size: 13px; fill: #1B3139; font-weight: 600; }
    .box-label-white { font-size: 13px; fill: #FFFFFF; font-weight: 700; }
    .small-mono { font-size: 11px; fill: #FFFFFF; font-family: 'Courier New', Courier, monospace; }
    .arrow-label { font-size: 10px; fill: #FF3621; font-weight: 700; font-family: 'Courier New', Courier, monospace; }
    .caption { font-size: 13px; fill: #1B3139; font-weight: 700; }
    .caption-sub { font-size: 11px; fill: #425563; font-style: italic; }
    .box-soft { fill: #F9F7F4; stroke: #1B3139; stroke-width: 1.5; }
    .box-dark { fill: #1B3139; stroke: #1B3139; stroke-width: 1.5; }
    .box-brick { fill: #FF3621; stroke: #FF3621; stroke-width: 1.5; }
    .box-future { fill: #F9F7F4; stroke: #425563; stroke-width: 1.5; stroke-dasharray: 5 3; }
    .arrow-gray { stroke: #425563; stroke-width: 1.5; fill: none; }
    .arrow-brick { stroke: #FF3621; stroke-width: 1.5; fill: none; }
    .divider { stroke: #425563; stroke-width: 0.5; stroke-dasharray: 4 4; }
  </style>
  <defs>
    <marker id="ah-gray" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto">
      <polygon points="0 0, 8 4, 0 8" fill="#425563"/>
    </marker>
    <marker id="ah-brick" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto">
      <polygon points="0 0, 8 4, 0 8" fill="#FF3621"/>
    </marker>
  </defs>

  <text x="500" y="32" class="title" text-anchor="middle">Per-table attachment vs catalog-scope policy</text>

  <line class="divider" x1="500" y1="55" x2="500" y2="450"/>

  <text x="245" y="75" class="panel-title" text-anchor="middle">PER-TABLE (TODAY)</text>
  <text x="245" y="95" class="panel-sub" text-anchor="middle">stable SQL DDL — works, doesn't scale</text>

  <rect class="box-dark" x="175" y="115" width="140" height="40" rx="6"/>
  <text x="245" y="140" class="box-label-white" text-anchor="middle">UDF: mask_ssn</text>

  <rect class="box-soft" x="80" y="220" width="160" height="36" rx="5"/>
  <text x="160" y="243" class="box-label" text-anchor="middle">transactions</text>

  <rect class="box-soft" x="80" y="270" width="160" height="36" rx="5"/>
  <text x="160" y="293" class="box-label" text-anchor="middle">employees</text>

  <rect class="box-soft" x="80" y="320" width="160" height="36" rx="5"/>
  <text x="160" y="343" class="box-label" text-anchor="middle">payroll_runs</text>

  <rect class="box-soft" x="80" y="370" width="160" height="36" rx="5"/>
  <text x="160" y="393" class="box-label" text-anchor="middle">new_customer_data</text>

  <path class="arrow-gray" d="M 230 155 C 215 185, 205 210, 195 218" marker-end="url(#ah-gray)"/>
  <path class="arrow-gray" d="M 245 155 C 235 215, 220 255, 205 268" marker-end="url(#ah-gray)"/>
  <path class="arrow-gray" d="M 260 155 C 255 245, 240 305, 220 318" marker-end="url(#ah-gray)"/>
  <path class="arrow-gray" d="M 275 155 C 290 280, 265 355, 235 368" marker-end="url(#ah-gray)"/>

  <text x="295" y="190" class="arrow-label">SET MASK</text>
  <text x="305" y="238" class="arrow-label">SET MASK</text>
  <text x="320" y="288" class="arrow-label">SET MASK</text>
  <text x="335" y="340" class="arrow-label">SET MASK</text>

  <text x="245" y="440" class="caption" text-anchor="middle">N tables = N attachments</text>
  <text x="245" y="458" class="caption-sub" text-anchor="middle">every new table is a SQL change</text>

  <text x="755" y="75" class="panel-title" text-anchor="middle">CATALOG-SCOPE (ABAC)</text>
  <text x="755" y="95" class="panel-sub" text-anchor="middle">attached once via Catalog Explorer / REST</text>

  <rect class="box-brick" x="600" y="115" width="310" height="50" rx="6"/>
  <text x="755" y="135" class="box-label-white" text-anchor="middle">POLICY: mask_ssn_policy</text>
  <text x="755" y="153" class="small-mono" text-anchor="middle">FOR TABLES MATCH COLUMNS has_tag_value('pii','ssn')</text>

  <rect class="box-soft" x="540" y="220" width="160" height="36" rx="5"/>
  <text x="620" y="243" class="box-label" text-anchor="middle">transactions</text>

  <rect class="box-soft" x="540" y="270" width="160" height="36" rx="5"/>
  <text x="620" y="293" class="box-label" text-anchor="middle">employees</text>

  <rect class="box-soft" x="540" y="320" width="160" height="36" rx="5"/>
  <text x="620" y="343" class="box-label" text-anchor="middle">payroll_runs</text>

  <rect class="box-soft" x="540" y="370" width="160" height="36" rx="5"/>
  <text x="620" y="393" class="box-label" text-anchor="middle">new_customer_data</text>

  <rect class="box-future" x="730" y="320" width="180" height="36" rx="5"/>
  <text x="820" y="343" class="box-label" text-anchor="middle">…every future table</text>

  <path class="arrow-brick" d="M 690 165 C 670 185, 650 210, 640 218" marker-end="url(#ah-brick)"/>
  <path class="arrow-brick" d="M 720 165 C 710 215, 690 255, 670 268" marker-end="url(#ah-brick)"/>
  <path class="arrow-brick" d="M 755 165 C 750 245, 730 305, 700 318" marker-end="url(#ah-brick)"/>
  <path class="arrow-brick" d="M 780 165 C 785 280, 770 355, 740 368" marker-end="url(#ah-brick)"/>
  <path class="arrow-brick" d="M 810 165 C 820 250, 820 300, 820 318" marker-end="url(#ah-brick)"/>

  <text x="755" y="440" class="caption" text-anchor="middle">1 policy = every tagged column</text>
  <text x="755" y="458" class="caption-sub" text-anchor="middle">current and future tables, no per-table SQL</text>
</svg>
""")

In [0]:
%sql
-- Attach per column on this table.  In production with catalog-scope
-- attachment, these two ALTERs would not be needed — the tag alone would
-- pull the mask in.
ALTER TABLE employees ALTER COLUMN ssn          SET MASK mask_ssn;
ALTER TABLE employees ALTER COLUMN home_address SET MASK mask_address;

### Step 6 — Watch the mask fire

In [0]:
%sql
-- If you're NOT in `hr_admins`, the SSN and address columns come back redacted.
-- If you ARE in `hr_admins`, you see raw values.
SELECT employee_id, full_name, banner, ssn, home_address
FROM   employees;

### Step 7 — Reusing the UDF on a brand-new table

The mask UDF is fully reusable — same `mask_ssn` function, attached to a *different* table's column. Two steps today (table-level mode); one step in catalog-scope mode (just the tag).

In [0]:
%sql
-- Brand-new payroll table — never touched our masks before
DROP TABLE IF EXISTS payroll_runs;
CREATE OR REPLACE TABLE payroll_runs (
  run_id       BIGINT,
  employee_id  STRING,
  employee_ssn STRING,
  gross_amount DECIMAL(10,2),
  run_date     DATE
);

INSERT INTO payroll_runs VALUES
  (1, 'EMP-08812', '111-22-3344', 1842.50, DATE '2026-05-09'),
  (2, 'EMP-04217', '222-33-4455', 2150.00, DATE '2026-05-09'),
  (3, 'EMP-01199', '333-44-5566', 1740.75, DATE '2026-05-09');

In [0]:
%sql
-- Tag the sensitive column.  In catalog-scope mode (UI/API), this single
-- line would be enough — the tag would pull `mask_ssn` in automatically.
ALTER TABLE payroll_runs ALTER COLUMN employee_ssn SET TAGS ('pii' = 'ssn');

In [0]:
%sql
-- Today's table-level DDL: we also have to attach.  The mask UDF, though,
-- is exactly the same one we wrote earlier — write once, attach anywhere.
ALTER TABLE payroll_runs ALTER COLUMN employee_ssn SET MASK mask_ssn;

In [0]:
%sql
-- Now mask_ssn fires on payroll_runs — same redaction logic, different table.
SELECT * FROM payroll_runs;

### The catalog-scope future state

In a workspace where the column-mask policy is attached at **catalog** scope (via the Catalog Explorer UI or REST API), Step 7 collapses to just the *tag* — the `ALTER TABLE ... SET MASK` disappears entirely, because the catalog-level policy says *"any column with tag `pii = ssn`, anywhere in this catalog, gets `mask_ssn`."*

That’s the operational payoff: **The UDF is written once, attaches the policy once, and every future table inherits the mask the moment a column is tagged.** No per-table SQL, no governance backlog when new tables land.

## Row filters work the same way

We won’t demo this live, but the shape is identical — only the UDF return type changes (`BOOLEAN` instead of the column’s type):

```sql
-- Sketch: each user only sees rows for their home banner
CREATE OR REPLACE FUNCTION banner_row_filter(row_banner STRING)
RETURNS BOOLEAN
RETURN
  is_account_group_member('all_banner_admins')
  OR row_banner = current_user_attribute('home_banner');

ALTER TABLE transactions
  SET ROW FILTER banner_row_filter ON (banner);
```

Same anatomy: scope, match, caller condition, transformation. The only difference is the UDF returns TRUE/FALSE instead of a masked value.

## Recap

In a short period of time, we went from raw tags to enforced ABAC:

- **Two policy types** — row filter and column mask — both are just SQL UDFs
- **Anatomy** — scope + match condition + caller condition + transformation
- **Catalog-scoped attachment** — the killer feature; one policy declaration covers every current and future table in the catalog that carries the matching tag
- **Auto-apply demo** — `payroll_runs` got masked without us ever telling UC about it specifically; the tag did the work

The combination of Section 2 (tagging hygiene) + Section 3 (policies that consume tags) is what makes ABAC operationally feasible at enterprise scale. Each new table inherits governance from the tags it carries — no manual per-table policy authoring, no role explosion.

# Section 4 — The Magic: Auto-Enforcement

Everything we’ve built so far — the tag vocabulary, the mask UDFs, the catalog-scope attachment — has been *setup*. This section is the payoff.

We’re going to pretend a brand-new table just landed from a legacy Oracle ERP migration. There is no policy for it. Nobody has wired up a mask. Just a new table, two tags, and a query — and we’ll watch what happens.

## What "auto-enforcement" actually means

In Section 3 the masks were attached **table by table** via `ALTER TABLE ... SET MASK`. That works, but it doesn’t scale — every new table is a per-table SQL change, and every per-table SQL change is a governance ticket.

The catalog-scope policy (attached once via *Catalog Explorer → Governance → Policies*) flips the model:

> *"Any column in this catalog tagged `pii = ssn` gets `mask_ssn` applied automatically — no per-table DDL required."*

The tag is the policy hook. The moment a column is tagged, the mask fires.

In [0]:
displayHTML("""
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 980 340" font-family="Helvetica, Arial, sans-serif">
  <style>
    .title { font-size: 22px; font-weight: 700; fill: #1B3139; }
    .step-label { font-size: 13px; font-weight: 700; fill: #1B3139; }
    .step-label-white { font-size: 13px; font-weight: 700; fill: #FFFFFF; }
    .step-sub { font-size: 11px; fill: #425563; font-style: italic; }
    .step-sub-white { font-size: 11px; fill: #FFFFFF; font-style: italic; opacity: 0.92; }
    .mono { font-size: 10px; fill: #1B3139; font-family: 'Courier New', Courier, monospace; }
    .mono-white { font-size: 10px; fill: #FFFFFF; font-family: 'Courier New', Courier, monospace; }
    .box-soft { fill: #F9F7F4; stroke: #1B3139; stroke-width: 1.5; }
    .box-dark { fill: #1B3139; stroke: #1B3139; stroke-width: 1.5; }
    .box-brick { fill: #FF3621; stroke: #FF3621; stroke-width: 2; }
    .arrow { stroke: #425563; stroke-width: 2; fill: none; }
    .caption-bottom { font-size: 14px; font-weight: 700; fill: #2EB67D; letter-spacing: 1.5px; }
    .caption-sub { font-size: 12px; fill: #425563; font-style: italic; }
  </style>
  <defs>
    <marker id="ah-step" markerWidth="10" markerHeight="10" refX="9" refY="5" orient="auto">
      <polygon points="0 0, 10 5, 0 10" fill="#425563"/>
    </marker>
  </defs>

  <text x="490" y="32" class="title" text-anchor="middle">Auto-enforcement: the tag is the policy hook</text>

  <rect class="box-soft" x="40" y="100" width="140" height="90" rx="6"/>
  <text x="110" y="125" class="step-label" text-anchor="middle">1. Table lands</text>
  <text x="110" y="143" class="step-sub" text-anchor="middle">from migration</text>
  <text x="110" y="170" class="mono" text-anchor="middle">CREATE TABLE …</text>

  <line class="arrow" x1="185" y1="145" x2="225" y2="145" marker-end="url(#ah-step)"/>

  <rect class="box-soft" x="235" y="100" width="140" height="90" rx="6"/>
  <text x="305" y="125" class="step-label" text-anchor="middle">2. Tag applied</text>
  <text x="305" y="143" class="step-sub" text-anchor="middle">by domain team</text>
  <text x="305" y="170" class="mono" text-anchor="middle">SET TAGS('pii','ssn')</text>

  <line class="arrow" x1="380" y1="145" x2="420" y2="145" marker-end="url(#ah-step)"/>

  <rect class="box-brick" x="430" y="92" width="140" height="106" rx="6"/>
  <text x="500" y="118" class="step-label-white" text-anchor="middle">3. Policy detects</text>
  <text x="500" y="138" class="step-sub-white" text-anchor="middle">automatically</text>
  <text x="500" y="165" class="mono-white" text-anchor="middle">has_tag_value</text>
  <text x="500" y="180" class="mono-white" text-anchor="middle">matches</text>

  <line class="arrow" x1="575" y1="145" x2="615" y2="145" marker-end="url(#ah-step)"/>

  <rect class="box-soft" x="625" y="100" width="140" height="90" rx="6"/>
  <text x="695" y="125" class="step-label" text-anchor="middle">4. Mask attaches</text>
  <text x="695" y="143" class="step-sub" text-anchor="middle">no SET MASK DDL</text>
  <text x="695" y="170" class="mono" text-anchor="middle">mask_ssn(col)</text>

  <line class="arrow" x1="770" y1="145" x2="810" y2="145" marker-end="url(#ah-step)"/>

  <rect class="box-dark" x="820" y="100" width="140" height="90" rx="6"/>
  <text x="890" y="125" class="step-label-white" text-anchor="middle">5. Query masked</text>
  <text x="890" y="143" class="step-sub-white" text-anchor="middle">next SELECT</text>
  <text x="890" y="170" class="mono-white" text-anchor="middle">XXX-XX-1234</text>

  <text x="490" y="265" class="caption-bottom" text-anchor="middle">ZERO ADDITIONAL GOVERNANCE WORK</text>
  <text x="490" y="290" class="caption-sub" text-anchor="middle">Admin team writes the UDF once; every future tagged column inherits the mask</text>
</svg>
""")

### One-time setup — catalog-scope policies

For the next cells to show pure auto-enforcement (tag → mask, no per-table `SET MASK` DDL), two catalog-scope policies need to be attached **once** on `:catalog`. The DDL below uses the `has_tag_value()` predicate so the policy matches only columns where `pii` has a specific value — `mask_ssn` fires for `pii = ssn`, `mask_address` fires for `pii = address`.

Run this cell once per workspace. After it's in place, every future table in the catalog inherits both masks the moment the right tag lands on a column.

In [0]:
%sql
-- One-time setup. Two notes before re-running this anywhere new:
--   1. The catalog name (`ademianczuk_uc_1_catalog`) is HARDCODED in
--      `ON CATALOG` and in the fully-qualified function names. UC does
--      not support `IDENTIFIER(:catalog)` for policy DDL, so the
--      `:catalog` / `:schema` widgets above do NOT propagate here.
--      Moving to another workspace? Edit the 4 occurrences below.
--   2. Function names MUST be fully qualified — the policy resolver
--      looks them up under `<catalog>.default` otherwise and fails.
CREATE OR REPLACE POLICY mask_ssn_policy
ON CATALOG IDENTIFIER(:catalog)
COMMENT 'Apply mask_ssn to columns tagged pii = ssn'
COLUMN MASK mask_ssn
TO `account users`
FOR TABLES MATCH COLUMNS has_tag_value('pii', 'ssn') AS ssn_col ON COLUMN ssn_col;

CREATE OR REPLACE POLICY mask_address_policy
ON CATALOG IDENTIFIER(:catalog)
COMMENT 'Apply mask_address to columns tagged pii = address'
COLUMN MASK mask_address
TO `account users`
FOR TABLES MATCH COLUMNS has_tag_value('pii', 'address') AS addr_col ON COLUMN addr_col;

In [0]:
%sql
-- Confirm both policies are attached
SHOW POLICIES ON CATALOG IDENTIFIER(:catalog);

## Demo — a brand new "ERP-migrated" table

In [0]:
%sql
-- Pretend this table just landed from a legacy Oracle ERP migration.
--
-- Note the explicit DROP: in UC, `CREATE OR REPLACE TABLE` preserves the
-- table's object identity, which means column tags survive the replace.
-- That would silently mask the "before tagging" SELECT below on a second
-- run.  DROP forces a true clean slate.
DROP TABLE IF EXISTS new_customer_data;
CREATE TABLE new_customer_data (
  customer_id     STRING,
  full_name       STRING,
  banner          STRING,
  ssn             STRING   COMMENT 'Customer SSN — landed from legacy ERP',
  home_address    STRING   COMMENT 'Customer home address — landed from legacy ERP',
  loyalty_tier    STRING,
  joined_date     DATE
);

In [0]:
%sql
INSERT INTO new_customer_data VALUES
  ('CUST-22001', 'Aisha Patel',     'FreshMart', '777-88-9911', '34 Birch St, Riverside CA',        'gold',     DATE '2023-04-12'),
  ('CUST-22002', 'Daniel Cho',      'Metro Kitchen',    '666-77-8800', '120 Elm St, Lakewood CA',     'platinum', DATE '2022-09-30'),
  ('CUST-22003', 'Hannah Williams', 'ValueMax',    '555-44-3322', '88 Main Rd, Greendale CA',      'silver',   DATE '2024-01-05'),
  ('CUST-22004', 'Raj Singh',       'Harvest Market', '444-33-2211', '12 Pine Rd, Pinecrest WA',          'gold',     DATE '2021-11-18'),
  ('CUST-22005', 'Megan O''Connor', 'Budget Basket', '333-22-1100', '901 Cedar Lane, Westfield WA', 'silver',   DATE '2025-02-22');

In [0]:
%sql
-- Right now the table is wide open.  No tags, no masks.  Anyone with SELECT
-- on this table sees raw SSN and address — exactly the kind of leakage
SELECT * FROM new_customer_data;

### The one-step fix: tag the columns

In [0]:
%sql
ALTER TABLE new_customer_data ALTER COLUMN ssn          SET TAGS ('pii' = 'ssn');
ALTER TABLE new_customer_data ALTER COLUMN home_address SET TAGS ('pii' = 'address');

In [0]:
%sql
-- Same SELECT.  No CREATE FUNCTION.  No SET MASK.  No new policy authoring.
-- The catalog-scope policy attached in Catalog Explorer saw the new tags
-- and applied mask_ssn / mask_address automatically.
SELECT * FROM new_customer_data;

> **This is what happens every time a new table lands and gets tagged. Zero additional governance work.**

## Why this generalizes — the migration math

The pattern just demonstrated is a single tag → instant governance. Multiply that by your legacy ERP migration backlog and the operational shape becomes clear.

| Per new table, RBAC world | Per new table, ABAC world |
|---|---|
| Identify sensitive columns manually | Auto-classification surfaces PII candidates within 24h |
| Mint roles for each access pattern | Subject attributes already populated from IdP groups |
| Author per-table grants / views | One catalog-scope policy, written once |
| Author per-column masks | Tag the column — mask attaches automatically |
| Re-review whenever the table evolves | Tag change is the only thing that needs to change |

The work doesn’t disappear — it **collapses upstream into tagging**, which has clearer owners and a much smaller surface area than per-table policy authoring.

### What scales for free

With Section 2 (auto-tagging + custom tags) + Section 3 (catalog-scope policies) in place, four things scale linearly with effort that used to scale combinatorially:

- **New tables** — pickup is automatic; the tag is the policy hook
- **New columns** on existing tables — same path; tag → masked
- **New consumers** — adding a user to `hr_admins` (or any caller group) toggles their view of every masked column at once
- **New sensitivity classes** — adding a new `pii` value (e.g. `pii = credit_card`) is one new UDF + one new policy attachment, and every column anywhere that adopts the tag inherits it

The thing that *doesn’t* scale for free is **tag hygiene** — but that’s an upstream problem the data classification agent + tag policies (Section 2 bonus) are designed to solve.

## The connection — governance that scales with the migration

There is a big backlog of legacy ERP workloads queued up for migration. In a per-table-policy world, every one of those migrations is also a governance project — author the masks, mint the roles, write the views, document the access patterns, hand off. The governance work becomes the bottleneck on the migration itself.

In the world we just demonstrated:

- The mask UDFs (`mask_ssn`, `mask_address`, future ones) are written **once**
- The catalog-scope attachment is set up **once** in Catalog Explorer
- Auto-classification handles the *what kind of data is this* tagging without human effort
- Domain teams own the *banner / owner / sensitivity* tagging for the tables they land

Every legacy table that lands with the right tags is **immediately governed**. The governance program stops being a tax on the migration and starts being a force multiplier on it.

> **Your governance scales with your migration, not against it.**

### Where it fails (honest)

ABAC at this scale has three failure modes worth naming upfront — better to plan for them than be surprised:

- **Untagged data** — if a sensitive column never gets tagged, it never gets masked. Mitigation: the classification agent catches well-known PII patterns; tag policies + the `data_owner` field route uncovered cases to humans who can fix them.
- **Tag drift** — someone applies `pii = SSN` (uppercase) and the mask doesn't fire because the policy matched the lowercase value. Mitigation: tag policies lock the vocabulary at the metastore level so the bad value is rejected at `ALTER TABLE` time.
- **Caller-condition complexity** — once policies blend multiple subject attributes (`is_account_group_member` AND `current_user_attribute`), the *why was this row returned?* question gets harder to answer at audit time. Mitigation: keep UDFs short; lean on the UC audit logs that record policy evaluation.

None of these are unique to ABAC — they're tradeoffs you accept in exchange for the migration math working.

## Recap

- **A brand-new table** landed (`new_customer_data` — pretend legacy ERP)
- **No policy authoring** happened
- **Two `ALTER COLUMN SET TAGS`** statements brought it under governance
- The catalog-scope policy saw the tags and applied the masks automatically
- The same playbook scales to every table in your migration backlog

### Fallback — if catalog-scope isn't set up yet

If the Catalog Explorer policy attachment isn't in place in this workspace, the demo above won't auto-mask. Drop in the per-table attachments below to still make the generalization point — *the UDFs are reused; only the wiring differs*:

```sql
ALTER TABLE new_customer_data ALTER COLUMN ssn          SET MASK mask_ssn;
ALTER TABLE new_customer_data ALTER COLUMN home_address SET MASK mask_address;
SELECT * FROM new_customer_data;
```

In production with catalog-scope attached, those two `SET MASK` lines disappear entirely — the tags are enough.

# Section 5 — Power BI Security

Everything we just demonstrated — row filters, column masks, catalog-scope policies — runs **server-side at Databricks**. That has a direct payoff for the BI tools sitting on top of UC, Power BI included. This section walks through what that means for a PBI Centre of Excellence.

## Architecture — what actually happens when PBI hits UC

Power BI in DirectQuery mode pushes the user’s query down to a Databricks SQL Warehouse. The warehouse asks Unity Catalog to evaluate the policies for the calling user. UC applies the row filter and the column mask **before the rows leave the warehouse**. Only the filtered, masked result is what travels back to PBI.

The thing to internalize: **Power BI never sees the raw data.** If the mask says redact, the redacted bytes are what PBI receives. There is no path where the unredacted value briefly exists on the BI side, no client-side step that could be misconfigured to leak it.

In [0]:
displayHTML("""
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 980 440" font-family="Helvetica, Arial, sans-serif">
  <style>
    .title { font-size: 22px; font-weight: 700; fill: #1B3139; }
    .stage-label { font-size: 14px; font-weight: 700; fill: #1B3139; }
    .stage-sub { font-size: 12px; fill: #425563; }
    .stage-label-white { font-size: 14px; font-weight: 700; fill: #FFFFFF; }
    .stage-sub-white { font-size: 12px; fill: #FFFFFF; opacity: 0.88; }
    .flow-label { font-size: 12px; fill: #425563; font-style: italic; }
    .flow-label-brick { font-size: 13px; fill: #FF3621; font-weight: 700; letter-spacing: 1px; }
    .boundary { font-size: 12px; fill: #FF3621; font-weight: 700; letter-spacing: 1.5px; }
    .box-soft { fill: #F9F7F4; stroke: #1B3139; stroke-width: 1.5; }
    .box-dark { fill: #1B3139; stroke: #1B3139; stroke-width: 1.5; }
    .box-brick { fill: #FF3621; stroke: #FF3621; stroke-width: 1.5; }
    .arrow-down { stroke: #425563; stroke-width: 2; fill: none; }
    .arrow-up { stroke: #2EB67D; stroke-width: 2; fill: none; }
    .boundary-line { stroke: #FF3621; stroke-width: 1.5; stroke-dasharray: 6 4; fill: none; }
  </style>
  <defs>
    <marker id="ah-d" markerWidth="10" markerHeight="10" refX="9" refY="5" orient="auto">
      <polygon points="0 0, 10 5, 0 10" fill="#425563"/>
    </marker>
    <marker id="ah-u" markerWidth="10" markerHeight="10" refX="9" refY="5" orient="auto">
      <polygon points="0 0, 10 5, 0 10" fill="#2EB67D"/>
    </marker>
  </defs>

  <text x="490" y="32" class="title" text-anchor="middle">Power BI → SQL Warehouse → Unity Catalog</text>

  <rect class="box-soft" x="260" y="60" width="220" height="60" rx="6"/>
  <text x="370" y="84" class="stage-label" text-anchor="middle">Power BI (DirectQuery)</text>
  <text x="370" y="104" class="stage-sub" text-anchor="middle">Entra-authenticated user</text>

  <line class="arrow-down" x1="370" y1="120" x2="370" y2="155" marker-end="url(#ah-d)"/>
  <text x="385" y="142" class="flow-label">SQL pushdown</text>

  <rect class="box-soft" x="260" y="160" width="220" height="60" rx="6"/>
  <text x="370" y="184" class="stage-label" text-anchor="middle">Databricks SQL Warehouse</text>
  <text x="370" y="204" class="stage-sub" text-anchor="middle">Asks UC to evaluate policies</text>

  <line class="arrow-down" x1="370" y1="220" x2="370" y2="255" marker-end="url(#ah-d)"/>

  <rect class="box-brick" x="260" y="260" width="220" height="60" rx="6"/>
  <text x="370" y="284" class="stage-label-white" text-anchor="middle">Unity Catalog</text>
  <text x="370" y="304" class="stage-sub-white" text-anchor="middle">Row filter + column mask applied</text>

  <line class="arrow-down" x1="370" y1="320" x2="370" y2="355" marker-end="url(#ah-d)"/>

  <rect class="box-dark" x="260" y="360" width="220" height="60" rx="6"/>
  <text x="370" y="384" class="stage-label-white" text-anchor="middle">Delta tables (ADLS / S3)</text>
  <text x="370" y="404" class="stage-sub-white" text-anchor="middle">Governed access only</text>

  <path class="arrow-up" d="M 530 390 C 620 390, 620 90, 530 90" marker-end="url(#ah-u)"/>
  <text x="700" y="200" class="flow-label-brick" text-anchor="middle">FILTERED + MASKED</text>
  <text x="700" y="220" class="flow-label-brick" text-anchor="middle">RESULT ONLY</text>

  <line class="boundary-line" x1="60" y1="142" x2="250" y2="142"/>
  <text x="155" y="132" class="boundary" text-anchor="middle">SECURITY BOUNDARY</text>
  <text x="155" y="158" class="flow-label" text-anchor="middle">raw data lives below this line</text>
</svg>
""")

## Identity flow — same person, same policies, every client

The user's identity travels with the query end-to-end:

- **Entra ID** authenticates the person opening the Power BI report
- **Databricks SSO** maps that Entra principal to a Databricks identity (the SCIM-synced user, with all their group memberships)
- **Unity Catalog** evaluates the row filter and column mask against that identity at query time

The practical consequence: `is_account_group_member('hr_admins')` returns the same answer whether the query came from a notebook, a SQL Editor tab, or a Power BI dashboard. One identity, one policy decision, regardless of the front door.

In [0]:
displayHTML("""
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 980 380" font-family="Helvetica, Arial, sans-serif">
  <style>
    .title { font-size: 22px; font-weight: 700; fill: #1B3139; }
    .center-label { font-size: 16px; font-weight: 700; fill: #FFFFFF; }
    .center-sub { font-size: 12px; fill: #FFFFFF; opacity: 0.88; }
    .client-label { font-size: 14px; font-weight: 700; fill: #1B3139; }
    .client-sub { font-size: 12px; fill: #425563; }
    .caption { font-size: 13px; fill: #425563; font-style: italic; }
    .box-soft { fill: #F9F7F4; stroke: #1B3139; stroke-width: 1.5; }
    .box-brick { fill: #FF3621; stroke: #FF3621; stroke-width: 1.5; }
    .arrow { stroke: #425563; stroke-width: 1.5; fill: none; }
  </style>
  <defs>
    <marker id="ah2" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto">
      <polygon points="0 0, 8 4, 0 8" fill="#425563"/>
    </marker>
  </defs>

  <text x="490" y="32" class="title" text-anchor="middle">Set once in UC — every client inherits</text>

  <rect class="box-brick" x="360" y="150" width="260" height="80" rx="6"/>
  <text x="490" y="180" class="center-label" text-anchor="middle">Unity Catalog</text>
  <text x="490" y="200" class="center-sub" text-anchor="middle">Row filters + column masks</text>
  <text x="490" y="216" class="center-sub" text-anchor="middle">defined once, evaluated per query</text>

  <rect class="box-soft" x="50" y="80" width="180" height="60" rx="6"/>
  <text x="140" y="105" class="client-label" text-anchor="middle">Databricks Notebook</text>
  <text x="140" y="124" class="client-sub" text-anchor="middle">SQL via cluster / warehouse</text>
  <line class="arrow" x1="230" y1="130" x2="360" y2="180" marker-end="url(#ah2)"/>

  <rect class="box-soft" x="50" y="240" width="180" height="60" rx="6"/>
  <text x="140" y="265" class="client-label" text-anchor="middle">SQL Editor</text>
  <text x="140" y="284" class="client-sub" text-anchor="middle">Direct SQL</text>
  <line class="arrow" x1="230" y1="260" x2="360" y2="210" marker-end="url(#ah2)"/>

  <rect class="box-soft" x="750" y="80" width="180" height="60" rx="6"/>
  <text x="840" y="105" class="client-label" text-anchor="middle">Power BI</text>
  <text x="840" y="124" class="client-sub" text-anchor="middle">DirectQuery (JDBC)</text>
  <line class="arrow" x1="750" y1="130" x2="620" y2="180" marker-end="url(#ah2)"/>

  <rect class="box-soft" x="750" y="240" width="180" height="60" rx="6"/>
  <text x="840" y="265" class="client-label" text-anchor="middle">Tableau / any JDBC</text>
  <text x="840" y="284" class="client-sub" text-anchor="middle">Same policy surface</text>
  <line class="arrow" x1="750" y1="260" x2="620" y2="210" marker-end="url(#ah2)"/>

  <text x="490" y="358" class="caption" text-anchor="middle">One identity (Entra → Databricks) → one policy decision, regardless of the client</text>
</svg>
""")

## Set once, persist everywhere

This is the operational shape the BI team should care about:

| Client | How it talks to UC | What it sees |
|---|---|---|
| Databricks notebook | SQL via cluster / warehouse | Same masks + filters |
| SQL Editor | Direct SQL on the warehouse | Same masks + filters |
| Power BI (DirectQuery) | JDBC over Entra-authenticated session | Same masks + filters |
| Tableau / any JDBC or ODBC client | Same JDBC surface | Same masks + filters |

Define the policy **once** in UC. Every current and future client inherits it. The BI tool is just another caller — its security story is the data layer’s security story.

## New in Runtime 18.1 — RLS-aware result caching

Historically, queries against tables with row filters or column masks **bypassed the result cache** — caching a per-user-filtered result for the wrong user would be a security incident, so the safe default was to skip caching entirely. The price was performance: every PBI page render re-evaluated the policies from scratch, even for the same user clicking the same visual twice.

Runtime 18.1 ships **policy-aware result caching**: the cache key now incorporates the evaluated policy and the calling principal. Same query, same principal, same policy state → cache hit. Different principal → cache miss, no leakage. The security model didn't change; the performance ceiling moved.

For a dashboard refreshing across 200 store managers, this is the difference between *acceptable* and *snappy*.

## What this means for the PBI CoE

Two shifts worth surfacing explicitly:

- **No more parallel RLS implementation in PBI.** RLS roles defined in Power BI semantic models, DAX security filter expressions, per-workspace security tables — none of that needs to exist when the row filter already lives in UC. The semantic model gets to be about *semantics*, not security.
- **One audit story.** *“Why did this user see that row?”* has one answer, not two — and that answer lives in UC’s audit logs, not split between PBI and the data layer.

> **The PBI team no longer needs to author PBI RLS rules for any table whose row filter is already enforced in UC.**

## Discussion

1. **How are you currently handling RLS in Power BI? Are you maintaining separate security logic per semantic model?**
2. **When a new dataset gets published to PBI, what's the process today to get it secured? Who owns it, how long does it take?**
3. **What would it feel like if all of that was handled once, at the data layer, before PBI ever touches it?**

## ⚠️ The one thing not to do

There's a tempting shortcut that breaks the whole model: giving Power BI **direct storage-level access** to ADLS or Blob — service principal credentials, account keys, SAS tokens, anything that lets the BI tool read the underlying files without going through the SQL Warehouse. **Don't.**

The moment PBI reads files directly from storage:

- UC is bypassed — no row filter, no column mask, no caller-condition check
- The masked values become the raw values, and PBI has no way to know the difference
- The audit trail goes dark — the security perimeter you just spent the day building no longer covers this access path

The only governed path is **through the SQL Warehouse**, which lives behind UC. Databricks and Microsoft engineering are actively working on richer integration paths — direct semantic model integration, Fabric/OneLake patterns — all of which preserve UC governance by design. If a use case appears to require direct-storage access, flag it and wait for the governed path. **Working around UC is never the right answer for a regulated dataset.**

## Recap

- **PBI inherits UC's security** via DirectQuery — server-side enforcement, raw data never crosses the boundary
- **One identity** travels Entra → Databricks → UC; one policy decision applies regardless of client
- **Set once, persist everywhere** — notebook, SQL Editor, PBI, any JDBC/ODBC client see the same masks and filters
- **Runtime 18.1 unlocks performance** with policy-aware result caching — no security tradeoff
- **One thing not to do**: never grant PBI storage-level access that bypasses UC

> **The PBI security conversation collapses from "what do we build in Power BI" to "what's already in UC." That's the whole pitch.**

# Conclusion — where the value compounds

We covered a lot of ground. Let me consolidate the arc into a shape you can act on Monday morning.

## What we built, in one breath

- **Section 1** — RBAC explodes combinatorially; ABAC scales because policy lives at *query time*, not at *assignment time*
- **Section 2** — Auto-classification plus custom governed tags give UC the resource attributes the policies need
- **Section 3** — Row filters and column masks are just SQL UDFs — `CASE WHEN` is the whole language
- **Section 4** — Catalog-scope policies turn the tag itself into the policy hook: tag a column, it's governed
- **Section 5** — Every client — notebook, SQL Editor, Power BI, any JDBC/ODBC tool — inherits the same enforcement, server-side

## Three teams, one model

The model we walked through serves three teams at once. Each owns a different piece, but they’re all operating on the same artifact:

| Team | What they own | What they stop maintaining |
|---|---|---|
| **Data governance team** | Tag policies, mask/filter UDFs, catalog-scope attachments | Per-table grants, per-banner views, role-explosion paperwork |
| **PBI CoE** | Semantic models, dashboards, distribution | Shadow RLS in PBI, parallel security tables, dual audit story |
| **Migration team** | Landing legacy tables into UC with the right tags | Governance backlog blocking each table landing |

## The bet

In a per-table-policy world, every one of those landings is its own governance project — author the masks, mint the roles, write the views, document the access patterns, hand off. **Governance becomes the schedule constraint on the migration itself.**

In the world we just demonstrated:

- The mask and filter UDFs are written **once**
- The catalog-scope policies are attached **once**
- Auto-classification surfaces PII without human effort
- Domain teams own the business tags for the tables they land
- Every downstream client — notebooks, PBI, JDBC — inherits the result automatically

Each legacy table that lands with the right tags is governed the moment it lands. The migration runs on its own clock.

> **Your governance program becomes a multiplier on the migration, not a tax on it.**

## Final Thoughts

Most security models fail at scale because the people maintaining them can’t keep up with the org’s complexity. ABAC works at enterprise scale because **it doesn’t ask them to.**

The policy is small. The attributes do the work. The audit story is one story. Every new table, every new banner, every new analyst inherits the model the moment they show up in the systems that already know about them — Entra, UC, the IdP, the classification agent.

> **Set the model once. Let the platform scale it.**